# Lesion Analysis

Analysis of the effects of cerebellar lesions on brain network dynamics, including:
- **Coherence analysis** between cortical regions under different lesion conditions
- **Power Spectral Density (PSD)** analysis for Purkinje cells and Deep Cerebellar Nuclei
- **Power ratio analysis** in the gamma and theta band

**Authors:** Alice Geminiani ([alice.geminiani@unipv.it](mailto:alice.geminiani@unipv.it)) and GitHub Copilot with Claude Opus 4.6

In [ ]:
import pickle
from pprint import pprint
import numpy as np
import xarray as xr
from utils.file_utils import *
from utils.plot_utils import *
import os
import pandas as pd

# Define the lesion group and output paths
lesion_group_name = "input_to_DCN_lesion"  # options: "input_to_DCN_lesion", "MLI_lesion", "input_to_DCN_and_MLI_lesion"
data_dir = "publication_data/NESTlesions/COSIM_NEST_LESIONrate_ampl_norm"         # options: ..COSIM_NEST_LESIONrate_ampl_norm, ..COSIM_NEST_LESIONratenorm, ..COSIM_NEST_LESIONno_norm
exp_data_dir = "/home/docker/packages/tvb-multiscale/rising_net/data/popa2013/"
# Create output directories if they don't exist
lesion_group_dir = os.path.join(data_dir, lesion_group_name)
if not os.path.exists(lesion_group_dir):
    os.makedirs(lesion_group_dir, exist_ok=True)

# Set output path for the coherence plots
output_path = os.path.join(lesion_group_dir, "coherence_lesions")

# Initialize a list to collect all statistical comparison results
all_statistical_results = []

# ============================================================================
# LOAD SAVED FDR-CORRECTED P-VALUES (if available)
# Always use input_to_DCN_and_MLI_lesion for FDR-corrected p-values (comprehensive analysis)
# ============================================================================
# Try versioned path first (SAVED FDR CORRECTION), then fall back to direct path
saved_stats_path = os.path.join(data_dir, 'input_to_DCN_and_MLI_lesion', 'SAVED FDR CORRECTION', 'statistical_comparisons.csv')
if not os.path.exists(saved_stats_path):
    saved_stats_path = os.path.join(data_dir, 'input_to_DCN_and_MLI_lesion', 'statistical_comparisons.csv')
saved_stats_df = None
if os.path.exists(saved_stats_path):
    saved_stats_df = pd.read_csv(saved_stats_path)
    print(f"Loaded saved statistical comparisons from: {saved_stats_path}")
    print(f"Number of saved comparisons: {len(saved_stats_df)}")
else:
    print(f"No saved statistical comparisons found at: {saved_stats_path}")
    print("Will compute p-values on-the-fly (without global FDR correction)")

def get_fdr_p_values(analysis_name, measure_name, condition_1='CONTROL'):
    """
    Look up FDR-corrected p-values from the saved CSV file.
    
    Parameters
    ----------
    analysis_name : str
        Name of the analysis (e.g., 'Trigeminal-Cerebellar Nuclei', 'Purkinje Cells')
    measure_name : str
        Name of the measure (e.g., 'Theta band', 'Gamma band')
    condition_1 : str
        Reference condition (default: 'CONTROL')
        
    Returns
    -------
    list
        List of FDR-corrected p-values for comparisons against condition_1
        Returns None if no saved data available
    """
    if saved_stats_df is None:
        return None
    
    # Filter for matching analysis and measure
    mask = (saved_stats_df['analysis'] == analysis_name) & \
           (saved_stats_df['measure'] == measure_name) & \
           (saved_stats_df['condition_1'] == condition_1)
    
    matching = saved_stats_df[mask]
    
    if len(matching) == 0:
        print(f"Warning: No saved p-values found for {analysis_name} - {measure_name}")
        return None
    
    # Return FDR-corrected p-values in order matching desired_conditions
    # Filter to only include conditions present in desired_conditions
    filtered_p_values = []
    for condition in desired_conditions:
        if condition == condition_1:
            continue
        row = matching[matching['condition_2'] == condition]
        if len(row) == 1:
            filtered_p_values.append(row['p_value_ttest_fdr'].values[0])
        else:
            print(f"Warning: No saved p-value for {analysis_name} - {measure_name}: {condition_1} vs {condition}")
            return None
    
    return filtered_p_values

# Load raw data
COH_raw = xr.DataArray.from_dict(load_pickled_dict(f"{data_dir}/COH.pkl"))
spikesPSD_raw = xr.DataArray.from_dict(load_pickled_dict(f"{data_dir}/SpikeRatesPSD.pkl"))

# ============================================================================
# HEMISPHERE AVERAGING: Average Left and Right hemisphere data for each simulation
# This ensures that each simulation provides one data point (not two per hemisphere)
# ============================================================================

# Average 
# across hemispheres
# spikesPSD has 'Hemisphere' dimension with 'Right' and 'Left'
spikesPSD = spikesPSD_raw.mean(dim='Hemisphere')
print(f"spikesPSD: Averaged across hemispheres. New shape: {spikesPSD.shape}")

# For COH, we need to average SAME-HEMISPHERE connections only
# Connection names are like: "Right Primary motor area - Right Whiskers"
# We only want to pair: Right-Right with Left-Left (same hemisphere)
# We exclude cross-hemisphere connections like "Right M1 - Left S1"

def get_hemisphere_prefix(region_name):
    """Extract the hemisphere prefix ('Right' or 'Left') from a region name."""
    if region_name.startswith('Right '):
        return 'Right'
    elif region_name.startswith('Left '):
        return 'Left'
    return None

def is_same_hemisphere_connection(conn_name):
    """Check if both parts of a connection are on the same hemisphere."""
    parts = conn_name.split(' - ')
    if len(parts) != 2:
        return False
    h1 = get_hemisphere_prefix(parts[0])
    h2 = get_hemisphere_prefix(parts[1])
    # Both must have a hemisphere prefix and they must match
    return h1 is not None and h2 is not None and h1 == h2

def get_base_connection_name(conn_name):
    """Remove Left/Right prefixes from both parts of a connection name."""
    parts = conn_name.split(' - ')
    cleaned_parts = []
    for part in parts:
        cleaned = part.replace('Right ', '').replace('Left ', '')
        cleaned_parts.append(cleaned)
    return ' - '.join(cleaned_parts)

def average_coh_hemispheres(coh_data):
    """Average coherence data across same-hemisphere connections only.
    
    Only pairs Right-Right connections with Left-Left connections.
    Cross-hemisphere connections (Right-Left, Left-Right) are excluded.
    """
    from collections import defaultdict
    
    connections = coh_data.coords['Connections'].values
    
    # Filter to only same-hemisphere connections
    same_hemi_conns = [c for c in connections if is_same_hemisphere_connection(c)]
    
    # Group same-hemisphere connections by their base name
    conn_groups = defaultdict(list)
    for conn in same_hemi_conns:
        base_name = get_base_connection_name(conn)
        conn_groups[base_name].append(conn)
    
    # Average each group (should be 2 connections: Right-Right and Left-Left)
    averaged_data_list = []
    
    for base_name, conn_list in conn_groups.items():
        # Average the Right-Right and Left-Left connections
        group_data = [coh_data.sel(Connections=conn) for conn in conn_list]
        avg_data = sum(group_data) / len(group_data)
        # Expand dimension back so we can concat
        avg_data = avg_data.expand_dims({'Connections': [base_name]})
        averaged_data_list.append(avg_data)
    
    # Concatenate all averaged data
    result = xr.concat(averaged_data_list, dim='Connections')
    
    return result

COH = average_coh_hemispheres(COH_raw)
print(f"COH: Averaged across hemispheres. New connections: {list(COH.coords['Connections'].values)}")

# Experimental data (unchanged)
COHexp = np.load(f"{exp_data_dir}/COH.npy")         # First element if with cerebON, second is with muscimol cerebOFF
freq_exp = np.arange(5.0, 48.0, 1.0)

# CerebOFF data
parent_dir = os.path.dirname(os.getcwd())
parent_of_current = os.path.abspath(os.path.join(os.getcwd(), '..'))
cerebON_OFFdir = os.path.join(parent_of_current, "Final10reps/COSIM_CEREBON_OFF")
COHcerebON_OFF_raw = xr.DataArray.from_dict(load_pickled_dict(f"{cerebON_OFFdir}/COH.pkl"))
PSDcerebON_OFF = xr.DataArray.from_dict(load_pickled_dict(f"{cerebON_OFFdir}/PSD.pkl"))

# Average cerebON_OFF across hemispheres too
COHcerebON_OFF = average_coh_hemispheres(COHcerebON_OFF_raw)

print(f"\nNew COH dimensions: {COH.dims}")
print(f"New spikesPSD dimensions: {spikesPSD.dims}")

# Set conditions based on lesion group
if lesion_group_name == "input_to_DCN_lesion":
    desired_conditions = ['CONTROL', 'PKJtoDCN', 'MOStoDCN']
elif lesion_group_name == "MLI_lesion":
    desired_conditions = ['CONTROL', 'INHtoPKJ', 'MLItoMLI']
elif lesion_group_name == "input_to_DCN_and_MLI_lesion":
    desired_conditions = ['CONTROL', 'PKJtoDCN', 'MOStoDCN', 'INHtoPKJ', 'MLItoMLI']   
else:
    raise ValueError(f"Unexpected lesion_group_name: {lesion_group_name}")

# Define frequency bands
freq_bands = [
    {
        'name': 'Theta band',
        'range': (6, 12)
    },
    {
        'name': 'Gamma band',
        'range': (25, 60)
    }
]

print(freq_bands[0]['range'][0]-freq_exp[0])

COHexp_theta = COHexp[0][int(freq_bands[0]['range'][0]-freq_exp[0]):int(freq_bands[0]['range'][1]-freq_exp[0]+1)]
COHexp_gamma = COHexp[0][int(freq_bands[1]['range'][0]-freq_exp[0]):int(freq_bands[1]['range'][1]-freq_exp[0]+1)]
COHexp_theta_cerebOFF = COHexp[1][int(freq_bands[0]['range'][0]-freq_exp[0]):int(freq_bands[0]['range'][1]-freq_exp[0]+1)]
COHexp_gamma_cerebOFF = COHexp[1][int(freq_bands[1]['range'][0]-freq_exp[0]):int(freq_bands[1]['range'][1]-freq_exp[0]+1)]

### Coherence plot

In [ ]:
# Define the full names (now hemisphere-averaged, so no Left/Right prefix)
PMDCN_full = 'Primary motor area - Primary somatosensory area, barrel field'

# Get shortened names for printing/reference
PMDCN_short = shorten_region_name(PMDCN_full)
print("Shortened name (for reference):", PMDCN_short)

print("\nLooking for this connection:", PMDCN_full)
print("\nChecking COH dimensions:", COH.dims)
print(f"\nTotal connections: {len(COH.coords['Connections'].values)}")

print("\nAttempting to access the data...")
# Use .sel() for proper dimension-based selection
COHpmdcn = COH.sel(Connections=PMDCN_full)
print("Success! Shape:", COHpmdcn.shape)
print(COHpmdcn)

In [ ]:
# Calculate mean coherence for different frequency bands
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.cm as cm

# Set default font sizes
set_default_plot_params()

# Get viridis colors
viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5),
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

# Define the two sets of connections (now hemisphere-averaged, single connection per type)
connection_sets = [
    {
        'name': 'Trigeminal-Cerebellar Nuclei',
        'connection': 'Principal sensory nucleus of the trigeminal - Cerebellar Nuclei',
        'colors': {
            'bar': viridis_colors['blue'],
            'points': viridis_colors['dark_purple']
        }
    },
    {
        'name': 'Sensory-Motor Cortex',
        'connection': 'Primary motor area - Primary somatosensory area, barrel field',
        'colors': {
            'bar': viridis_colors['green'],
            'points': viridis_colors['teal']
        }
    }
]

# Title mapping for cleaner labels
titles = {'Trigeminal-Cerebellar Nuclei': 'PrV-CN',
          'Sensory-Motor Cortex': 'M1-S1'}

# Create figure with 3x2 subplots (added row for gamma/theta ratio)
fig, axes = plt.subplots(3, 2, figsize=(10, 15), constrained_layout=True)
fig.suptitle('Coherence by Lesion Condition', y=1.02, fontsize=18)

# Store coherence data for ratio calculation
coherence_data = {}

# Process each frequency band and connection set
for freq_idx, freq_band in enumerate(freq_bands):
    for conn_idx, conn_set in enumerate(connection_sets):
      
        # Get data for this connection (already hemisphere-averaged)
        # Use .sel() for dimension-based selection
        data = COH.sel(Connections=conn_set['connection'])
        
        # Select frequency range
        freq_range = data.sel(Frequency=slice(freq_band['range'][0], freq_band['range'][1]))
        
        # Calculate means across frequencies
        freq_mean = freq_range.mean(dim='Frequency')
        
        # Store data for ratio calculation
        if conn_set['name'] not in coherence_data:
            coherence_data[conn_set['name']] = {}
        coherence_data[conn_set['name']][freq_band['name']] = freq_mean
        
        # Compute stats (data is already hemisphere-averaged at loading)
        means, sems, individual_points = compute_condition_stats(freq_mean, desired_conditions)
        
        ax = axes[freq_idx, conn_idx]
        
        # Try to get FDR-corrected p-values from saved CSV
        fdr_p_values = get_fdr_p_values(conn_set['name'], freq_band['name'])
        
        if fdr_p_values is not None:
            print(f"Using saved FDR-corrected p-values for {conn_set['name']} - {freq_band['name']}: {fdr_p_values}")
            p_values_to_use = fdr_p_values
        else:
            # Compute raw p-values as fallback
            reference_idx = 0
            comparisons = [(reference_idx, i) for i in range(len(desired_conditions)) if i != reference_idx]
            p_values_to_use = []
            for c1, c2 in comparisons:
                t_stat, p_val = stats.ttest_ind(individual_points[c1], individual_points[c2])
                p_values_to_use.append(p_val)
            print(f"Using raw p-values (no saved FDR data) for {conn_set['name']} - {freq_band['name']}: {p_values_to_use}")
        
        # Plot with p-values (FDR-corrected if available)
        bar_plot_with_stats(
            ax, means, sems, individual_points, desired_conditions,
            bar_color=conn_set['colors']['bar'], 
            point_color=conn_set['colors']['points'],
            ylabel=f'Coherence\n{freq_band["name"]}',
            title=f'{titles[conn_set["name"]]}',
            ylim_bottom=0.5 if conn_set['name'] == 'Trigeminal-Cerebellar Nuclei' else None,
            p_values=p_values_to_use
        )
        
        # Print statistics and collect results
        print_condition_stats(conn_set['name'], freq_band['name'], freq_band['range'],
                              desired_conditions, means, sems, individual_points)
        stats_results = print_statistical_tests(conn_set['name'], freq_band['name'], desired_conditions, individual_points)
        all_statistical_results.extend(stats_results)

# Calculate gamma/theta ratio
for conn_idx, conn_set in enumerate(connection_sets):
    # Get gamma and theta data
    gamma_data = coherence_data[conn_set['name']]['Gamma band']
    theta_data = coherence_data[conn_set['name']]['Theta band']
    
    # Calculate ratio
    ratio_data = gamma_data / theta_data
    
    # Compute stats (data is already hemisphere-averaged)
    means, sems, individual_points = compute_condition_stats(ratio_data, desired_conditions)
    
    ax = axes[2, conn_idx]  # Third row
    
    # Try to get FDR-corrected p-values from saved CSV
    fdr_p_values = get_fdr_p_values(conn_set['name'], 'Gamma/Theta Ratio')
    
    if fdr_p_values is not None:
        print(f"Using saved FDR-corrected p-values for {conn_set['name']} - Gamma/Theta Ratio: {fdr_p_values}")
        p_values_to_use = fdr_p_values
    else:
        # Compute raw p-values as fallback
        reference_idx = 0
        comparisons = [(reference_idx, i) for i in range(len(desired_conditions)) if i != reference_idx]
        p_values_to_use = []
        for c1, c2 in comparisons:
            t_stat, p_val = stats.ttest_ind(individual_points[c1], individual_points[c2])
            p_values_to_use.append(p_val)
        print(f"Using raw p-values (no saved FDR data) for {conn_set['name']} - Gamma/Theta Ratio: {p_values_to_use}")
    
    # Plot with p-values (FDR-corrected if available)
    bar_plot_with_stats(
        ax, means, sems, individual_points, desired_conditions,
        bar_color=conn_set['colors']['bar'], 
        point_color=conn_set['colors']['points'],
        ylabel=f'Gamma/Theta\nCoherence Ratio',
        title=f'{titles[conn_set["name"]]}',
        ylim_bottom=0.5 if conn_set['name'] == 'Trigeminal-Cerebellar Nuclei' else None,
        p_values=p_values_to_use
    )
    
    # Print statistics and collect results
    print_condition_stats(conn_set['name'], 'Gamma/Theta Ratio', (0, 0),
                          desired_conditions, means, sems, individual_points)
    stats_results = print_statistical_tests(conn_set['name'], 'Gamma/Theta Ratio', desired_conditions, individual_points)
    all_statistical_results.extend(stats_results)

plt.tight_layout()

# Save the figure
save_figure_multi_format(fig, output_path)

plt.show()

### Power Spectral Density Analysis for Purkinje Cells and Deep Cerebellar Nuclei

In [ ]:
# Function to get the data for each population (hemisphere-averaged at loading)
def get_population_data(pop_name, condition):
    # Get PSD data including all repetitions (already hemisphere-averaged)
    psd_data = spikesPSD.sel(
        **{'Lesion Condition': condition,
           'Population - Region': pop_name}
    ).values
    return psd_data

# Check available population labels
print("Available populations:")
for label in spikesPSD.coords['Population - Region'].values:
    print(f"- {label}")

print(f"\nspikesPSD dimensions: {spikesPSD.dims}")

# Test data access for a single population
test_pop = 'purkinje_cell - Al'
test_data = get_population_data(test_pop, 'CONTROL')
print(f"\nShape of test data for {test_pop}:")
print(f"(Repetitions × Frequencies): {test_data.shape}")

In [ ]:
# Plot average PSD for Purkinje cells and CN
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import matplotlib.cm as cm

# Set default font sizes
set_default_plot_params()

# === Toggle PSD normalization ===
normalize_psd = False  # Set to True to normalize PSD by total power

# Get frequency values (same for all conditions)
freqs = spikesPSD.coords['frequency'].values

# Get viridis colors
viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5), 
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

# Function to plot PSD for all populations on the same subplot
def plot_all_populations_psd(ax, normalize = False):
    # Only plot CONTROL condition
    condition = 'CONTROL'
    
    # Define populations with their viridis colors
    populations = {
        'Mossy Fibers': {'name': 'mossy_fibers - Al', 'color': viridis_colors['dark_purple'], 'linestyle': '--'},
        'Purkinje Cells': {'name': 'purkinje_cell - Al', 'color': viridis_colors['teal'], 'linestyle': '-'},
        'CNe': {'name': 'dcn_cell_glut_large - CN', 'color': viridis_colors['blue'], 'linestyle': '-'}
    }
    
    for pop_label, pop_info in populations.items():
        pop_name = pop_info['name']
        color = pop_info['color']
        linestyle = pop_info['linestyle']
        
        # Get PSD data (already hemisphere-averaged)
        psd_data = spikesPSD.sel(
            **{'Lesion Condition': condition, 
               'Population - Region': pop_name}
        ).values
        
        if psd_data.size == 0:
            print(f"Warning: No data found for {pop_name} in CONTROL condition")
            continue
        
        if normalize:
            psd_data = psd_data / np.sum(psd_data, axis=1, keepdims=True)  # Normalize each trial by total power
        
        # Calculate mean and STD across repetitions
        mean_psd = np.mean(psd_data, axis=0)
        std_psd = np.std(psd_data, axis=0)
        
        # Plot population PSD
        ax.plot(freqs, mean_psd, label=pop_label, 
               color=color, linewidth=3, linestyle=linestyle)
        ax.fill_between(freqs, mean_psd - std_psd, mean_psd + std_psd, 
                     alpha=0.3, color=color)

    # Customize subplot
    ax.set_xlabel('Frequency (Hz)', fontsize=14)
    ylabel = 'Normalized PSD' if normalize else 'Power Spectral Density'
    ax.set_ylabel(ylabel, fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlim(1, 100)
    
    # Add vertical lines for frequency bands using viridis colors
    ax.axvline(x=6, color=viridis_colors['yellow'], linestyle=':', linewidth=2, alpha=0.7)
    ax.axvline(x=12, color=viridis_colors['yellow'], linestyle=':', linewidth=2, alpha=0.7)
    ax.axvline(x=25, color=viridis_colors['green'], linestyle=':', linewidth=2, alpha=0.7)
    ax.axvline(x=60, color=viridis_colors['green'], linestyle=':', linewidth=2, alpha=0.7)
    
    # Add shaded regions for frequency bands
    ax.axvspan(6, 12, alpha=0.1, color=viridis_colors['yellow'], label='Theta')
    ax.axvspan(25, 60, alpha=0.1, color=viridis_colors['green'], label='Gamma')
    
    ax.legend(fontsize=10, loc='upper left')
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=12)

# ============================================================================
# Helper functions for band power computation
# Consistent with plot_differential_gamma_bars in the power ratio analysis cell.
# spikesPSD is already hemisphere-averaged: spikesPSD = spikesPSD_raw.mean(dim='Hemisphere')
# so each trial is the mean of Left and Right hemisphere values.
# ============================================================================

def compute_band_power(pop_name, condition, freq_range, normalize=False):
    """Compute integrated power in a frequency band for each trial.
    Data is already hemisphere-averaged (spikesPSD = spikesPSD_raw.mean(dim='Hemisphere')).
    """
    band_freqs = spikesPSD.sel(frequency=slice(freq_range[0], freq_range[1])).coords['frequency'].values
    psd_data = spikesPSD.sel(
        **{'Lesion Condition': condition,
           'Population - Region': pop_name,
           'frequency': slice(freq_range[0], freq_range[1])}
    ).values
    if normalize:
        # Normalize by total power across all frequencies for each trial
        full_psd = spikesPSD.sel(
            **{'Lesion Condition': condition,
               'Population - Region': pop_name}
        ).values
        total_power = np.sum(full_psd, axis=1, keepdims=True)
        psd_data = psd_data / total_power
    return np.array([np.trapz(trial_psd, band_freqs) for trial_psd in psd_data])


def compute_power_ratio(pop_name, ref_name, condition, freq_range, normalize=False):
    """Compute log10 power ratio relative to reference population for each trial."""
    pop_power = compute_band_power(pop_name, condition, freq_range, normalize=normalize)
    ref_power = compute_band_power(ref_name, condition, freq_range, normalize=normalize)
    return np.log10(pop_power / ref_power)


# Function to create bar plot comparing theta and gamma band differences
def plot_band_differences(ax, normalize=False):
    theta_range = (6, 12)
    gamma_range = (25, 60)
    
    populations = {
        'Purkinje Cells': {'name': 'purkinje_cell - Al', 'color': viridis_colors['teal']},
        'CNe': {'name': 'dcn_cell_glut_large - CN', 'color': viridis_colors['blue']}
    }
    
    mossy_name = 'mossy_fibers - Al'
    condition = 'CONTROL'
    
    # Statistical analysis: compare raw power between populations
    norm_label = " (Normalized)" if normalize else ""
    print(f"\n=== Statistical Analysis: Raw Power Comparisons{norm_label} ===")
    for band, freq_range in [('theta', theta_range), ('gamma', gamma_range)]:
        print(f"\n{band.upper()} BAND {freq_range} Hz:")
        mossy_power = compute_band_power(mossy_name, condition, freq_range, normalize=normalize)
        for pop_label, pop_info in populations.items():
            pop_power = compute_band_power(pop_info['name'], condition, freq_range, normalize=normalize)
            t_stat, p_val = stats.ttest_ind(mossy_power, pop_power)
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  Mossy vs {pop_label}: t={t_stat:.3f}, p={p_val:.4f} [{sig}]")
    
    # Compute power ratios for bar plot using compute_power_ratio
    results = {}
    for pop_label, pop_info in populations.items():
        results[pop_label] = {
            'theta': compute_power_ratio(pop_info['name'], mossy_name, condition, theta_range, normalize=normalize),
            'gamma': compute_power_ratio(pop_info['name'], mossy_name, condition, gamma_range, normalize=normalize)
        }
    
    # Create bar plot with viridis colors
    x_pos = np.arange(len(populations))
    width = 0.35
    
    theta_means = [np.mean(results[pop]['theta']) for pop in populations.keys()]
    theta_stds = [np.std(results[pop]['theta']) for pop in populations.keys()]
    gamma_means = [np.mean(results[pop]['gamma']) for pop in populations.keys()]
    gamma_stds = [np.std(results[pop]['gamma']) for pop in populations.keys()]
    
    ax.bar(x_pos - width/2, theta_means, width, yerr=theta_stds, 
           label='Theta', color=viridis_colors['yellow'], alpha=0.8, capsize=5)
    ax.bar(x_pos + width/2, gamma_means, width, yerr=gamma_stds,
           label='Gamma', color=viridis_colors['green'], alpha=0.8, capsize=5)
    
    for i, (theta_mean, gamma_mean) in enumerate(zip(theta_means, gamma_means)):
        ax.text(i - width/2, theta_mean + theta_stds[i] + 0.02, f'{theta_mean:.2f}±{theta_stds[i]:.2f}',
                ha='center', va='bottom', fontsize=12)
        ax.text(i + width/2, gamma_mean + gamma_stds[i] + 0.02, f'{gamma_mean:.2f}±{gamma_stds[i]:.2f}',
                ha='center', va='bottom', fontsize=12)
    
    ax.set_xlabel('Population', fontsize=14)
    ylabel = 'Log₁₀(Norm. Power Ratio)' if normalize else 'Log₁₀(Population/Mossy Power Ratio)'
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_xticks(x_pos)
    ax.set_yticks(np.arange(-1.0, 2.0, 0.5))
    ax.set_ylim(-1.0, 2.0)
    ax.set_xticklabels(populations.keys())
    ax.legend(fontsize=10, loc='lower left')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax.tick_params(axis='both', which='major', labelsize=12)

# Set up the plot
fig = plt.figure(figsize=(10, 5))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1])
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

plot_all_populations_psd(ax1, normalize=normalize_psd)
plot_band_differences(ax2, normalize=normalize_psd)

norm_suffix = '_normalized' if normalize_psd else ''
plt.tight_layout()
save_figure_multi_format(fig, os.path.join(lesion_group_dir, f'PSD_comparison_all{norm_suffix}'))
plt.show()

In [ ]:
# Plot average PSD for Mossy Fibers and Granule Cells + power ratio
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import matplotlib.cm as cm

set_default_plot_params()

freqs = spikesPSD.coords['frequency'].values

viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5),
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

def plot_mossy_granule_psd(ax, normalize=False):
    condition = 'CONTROL'
    populations = {
        'Mossy Fibers': {'name': 'mossy_fibers - Al', 'color': viridis_colors['dark_purple'], 'linestyle': '--'},
        'Granule Cells': {'name': 'granule_cell - Al', 'color': viridis_colors['teal'], 'linestyle': '-'}
    }

    for pop_label, pop_info in populations.items():
        pop_name = pop_info['name']
        color = pop_info['color']
        linestyle = pop_info['linestyle']

        psd_data = spikesPSD.sel(
            **{'Lesion Condition': condition,
               'Population - Region': pop_name}
        ).values

        if psd_data.size == 0:
            print(f"Warning: No data found for {pop_name} in CONTROL condition")
            continue

        if normalize:
            psd_data = psd_data / np.sum(psd_data, axis=1, keepdims=True)

        mean_psd = np.mean(psd_data, axis=0)
        std_psd = np.std(psd_data, axis=0)

        ax.plot(freqs, mean_psd, label=pop_label,
               color=color, linewidth=3, linestyle=linestyle)
        ax.fill_between(freqs, mean_psd - std_psd, mean_psd + std_psd,
                     alpha=0.3, color=color)

    ax.set_xlabel('Frequency (Hz)', fontsize=14)
    ylabel = 'Normalized PSD' if normalize else 'Power Spectral Density'
    ax.set_ylabel(ylabel, fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlim(1, 100)

    ax.axvline(x=6, color=viridis_colors['yellow'], linestyle=':', linewidth=2, alpha=0.7)
    ax.axvline(x=12, color=viridis_colors['yellow'], linestyle=':', linewidth=2, alpha=0.7)
    ax.axvline(x=25, color=viridis_colors['green'], linestyle=':', linewidth=2, alpha=0.7)
    ax.axvline(x=60, color=viridis_colors['green'], linestyle=':', linewidth=2, alpha=0.7)

    ax.axvspan(6, 12, alpha=0.1, color=viridis_colors['yellow'], label='Theta')
    ax.axvspan(25, 60, alpha=0.1, color=viridis_colors['green'], label='Gamma')

    ax.legend(fontsize=10, loc='upper left')
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=12)


def plot_granule_band_differences(ax, normalize=False):
    theta_range = (6, 12)
    gamma_range = (25, 60)

    populations = {
        'Granule Cells': {'name': 'granule_cell - Al', 'color': viridis_colors['teal']}
    }

    mossy_name = 'mossy_fibers - Al'
    condition = 'CONTROL'

    norm_label = " (Normalized)" if normalize else ""
    print(f"\n=== Statistical Analysis: Mossy vs Granule Raw Power{norm_label} ===")
    for band, freq_range in [('theta', theta_range), ('gamma', gamma_range)]:
        print(f"\n{band.upper()} BAND {freq_range} Hz:")
        mossy_power = compute_band_power(mossy_name, condition, freq_range, normalize=normalize)
        for pop_label, pop_info in populations.items():
            pop_power = compute_band_power(pop_info['name'], condition, freq_range, normalize=normalize)
            t_stat, p_val = stats.ttest_ind(mossy_power, pop_power)
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  Mossy vs {pop_label}: t={t_stat:.3f}, p={p_val:.4f} [{sig}]")

    results = {}
    for pop_label, pop_info in populations.items():
        results[pop_label] = {
            'theta': compute_power_ratio(pop_info['name'], mossy_name, condition, theta_range, normalize=normalize),
            'gamma': compute_power_ratio(pop_info['name'], mossy_name, condition, gamma_range, normalize=normalize)
        }

    x_pos = np.arange(len(populations))
    width = 0.35

    theta_means = [np.mean(results[pop]['theta']) for pop in populations.keys()]
    theta_stds = [np.std(results[pop]['theta']) for pop in populations.keys()]
    gamma_means = [np.mean(results[pop]['gamma']) for pop in populations.keys()]
    gamma_stds = [np.std(results[pop]['gamma']) for pop in populations.keys()]

    ax.bar(x_pos - width/2, theta_means, width, yerr=theta_stds,
           label='Theta', color=viridis_colors['yellow'], alpha=0.8, capsize=5)
    ax.bar(x_pos + width/2, gamma_means, width, yerr=gamma_stds,
           label='Gamma', color=viridis_colors['green'], alpha=0.8, capsize=5)

    for i, (theta_mean, gamma_mean) in enumerate(zip(theta_means, gamma_means)):
        ax.text(i - width/2, theta_mean + theta_stds[i] + 0.02, f'{theta_mean:.2f}±{theta_stds[i]:.2f}',
                ha='center', va='bottom', fontsize=12)
        ax.text(i + width/2, gamma_mean + gamma_stds[i] + 0.02, f'{gamma_mean:.2f}±{gamma_stds[i]:.2f}',
                ha='center', va='bottom', fontsize=12)

    ax.set_xlabel('Population', fontsize=14)
    ylabel = 'Log₁₀(Norm. Power Ratio)' if normalize else 'Log₁₀(Granule/Mossy Power Ratio)'
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_xticks(x_pos)
    ax.set_yticks(np.arange(-1.0, 2.0, 0.5))
    ax.set_ylim(-1.0, 2.0)
    ax.set_xticklabels(populations.keys())
    ax.legend(fontsize=10, loc='lower left')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax.tick_params(axis='both', which='major', labelsize=12)

# Set up the plot
fig = plt.figure(figsize=(10, 5))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1])
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

plot_mossy_granule_psd(ax1, normalize=normalize_psd)
plot_granule_band_differences(ax2, normalize=normalize_psd)

norm_suffix = '_normalized' if normalize_psd else ''
plt.tight_layout()
save_figure_multi_format(fig, os.path.join(lesion_group_dir, f'PSD_comparison_mossy_granule{norm_suffix}'))
plt.show()

In [ ]:
# Create first figure for PC and CN gamma band analysis with 3x2 subplots
fig1, axes1 = plt.subplots(3, 2, figsize=(10, 15))

# Set default font sizes
set_default_plot_params()

# Define gamma band range
gamma_range = (25, 60)

# Get viridis colors
viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5),
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

def plot_gamma_bars(pop_name, ax, color, title):
    """Plot gamma band power bars for a population.
    Data is already hemisphere-averaged at loading.
    """
    means = []
    sems = []
    individual_points = []
    all_data = {}
    
    # Get frequency values for integration
    freqs = spikesPSD.sel(frequency=slice(gamma_range[0], gamma_range[1])).coords['frequency'].values
    
    for condition in desired_conditions:
        # Get PSD data (already hemisphere-averaged)
        condition_psd = spikesPSD.sel(
            **{'Lesion Condition': condition, 
               'Population - Region': pop_name,
               'frequency': slice(gamma_range[0], gamma_range[1])}
        ).values
        
        # Calculate total power using trapezoid integration for each trial
        trial_powers = np.array([np.trapz(trial_psd, freqs) for trial_psd in condition_psd])
        
        means.append(np.mean(trial_powers))
        sems.append(np.std(trial_powers) / np.sqrt(len(trial_powers)))  # SEM
        individual_points.append(trial_powers)
        all_data[condition] = trial_powers
    
    means = np.array(means)
    sems = np.array(sems)
    
    # Try to get FDR-corrected p-values from saved CSV
    fdr_p_values = get_fdr_p_values(title, f'Gamma Power ({gamma_range[0]}-{gamma_range[1]} Hz)')
    
    if fdr_p_values is not None:
        print(f"Using saved FDR-corrected p-values for {title}: {fdr_p_values}")
    else:
        print(f"No saved FDR p-values found for {title} - using raw p-values for annotations")
    
    # Create bar plot with stats (use FDR p-values if available)
    bar_plot_with_stats(
        ax, means, sems, individual_points, desired_conditions,
        bar_color=color, point_color=color,
        point_size=20, point_alpha=0.6,
        ylabel='Total Power in Gamma Band',
        title=title,
        add_stat_annotations=True,
        p_values=fdr_p_values  # Pass FDR-corrected p-values if available
    )
    
    # Print statistical results with Cohen's d and collect results
    stats_results = print_statistical_tests(title, f'Gamma Power ({gamma_range[0]}-{gamma_range[1]} Hz)', 
                                            desired_conditions, individual_points)
    all_statistical_results.extend(stats_results)
    
    return all_data

def get_coherence_data(connection, gamma_range):
    """Get coherence data organized by condition (already hemisphere-averaged)."""
    coherence_data = {}
    
    for condition in desired_conditions:
        coh_data = COH.sel(
            **{'Lesion Condition': condition,
               'Connections': connection,
               'Frequency': slice(gamma_range[0], gamma_range[1])}
        ).values
        
        coh_mean = np.mean(coh_data, axis=1)  # Average across frequencies
        coherence_data[condition] = coh_mean
    
    return coherence_data

def plot_gamma_correlation(power_data, coh_data, ax, title):
    """Plot correlation between power and coherence data."""
    x_vals = []
    y_vals = []
    
    for condition in desired_conditions:
        x_vals.extend(power_data[condition])
        y_vals.extend(coh_data[condition])
    
    x_vals = np.array(x_vals)
    y_vals = np.array(y_vals)
    
    r, p_val = stats.pearsonr(x_vals, y_vals)
    
    ax.scatter(x_vals, y_vals, alpha=0.6, color=viridis_colors['teal'])
    
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, p(x_line), '--', alpha=0.8, color=viridis_colors['green'])
    
    ax.set_xlabel('Total Power in Gamma Band')
    ax.set_ylabel('Coherence in Gamma Band')
    ax.set_title(f'{title}\nr={r:.3f}, p={p_val:.4f}')
    
    return ax

# Cell population names
pc_pop = 'purkinje_cell - Al'
cn_pop = 'dcn_cell_glut_large - CN'
mossy_pop = 'mossy_fibers - Al'

# Create gamma band plots for PC and CN (first row) using viridis colors
pc_data = plot_gamma_bars(pc_pop, axes1[0,0], viridis_colors['teal'], 'Purkinje Cells Power in Gamma Band')
cn_data = plot_gamma_bars(cn_pop, axes1[0,1], viridis_colors['blue'], 'Cerebellar Nuclei Power in Gamma Band')

# Get coherence data (single connection per type, already hemisphere-averaged)
trig_cn_coh = get_coherence_data(connection_sets[0]['connection'], gamma_range)
sens_mot_coh = get_coherence_data(connection_sets[1]['connection'], gamma_range)
print("sens_mot_coh: ", sens_mot_coh)

# Column 1: All Purkinje cell-related correlations
plot_gamma_correlation(pc_data, trig_cn_coh, axes1[1,0], 
                'Cerebellar Input-Output Coherence\nvs Purkinje Cell Power')
plot_gamma_correlation(pc_data, sens_mot_coh, axes1[2,0], 
                'M1-S1 Cortex Coherence\nvs Purkinje Cell Power')

# Column 2: All Cerebellar Nuclei-related correlations
plot_gamma_correlation(cn_data, trig_cn_coh, axes1[1,1], 
                'Cerebellar Input-Output Coherence\nvs Cerebellar Nuclei Power')
plot_gamma_correlation(cn_data, sens_mot_coh, axes1[2,1], 
                'M1-S1 Cortex Coherence\nvs Cerebellar Nuclei Power')

fig1.suptitle('Power Ratio Analysis in Gamma Band', y=1.02, fontsize=18)
plt.tight_layout()
save_figure_multi_format(fig1, os.path.join(lesion_group_dir, 'PSD_gamma_analysis'))

# Create second figure for Mossy Fiber analysis
fig2, ax2 = plt.subplots(figsize=(10, 8))
mossy_data = plot_gamma_bars(mossy_pop, ax2, viridis_colors['dark_purple'], 'Mossy Fibers Power in Gamma Band')
fig2.suptitle('Mossy Fibers Power in Gamma Band', fontsize=18)
plt.tight_layout()
save_figure_multi_format(fig2, os.path.join(lesion_group_dir, 'PSD_gamma_analysis_mossy'))

plt.show()

In [ ]:
# Create figure for differential gamma band analysis with 3x2 subplots
fig, axes = plt.subplots(3, 2, figsize=(10, 15))

# Set default font sizes
set_default_plot_params()

# Get viridis colors
viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5),
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

def plot_differential_gamma_bars(pop_name, ax, color, title, reference_pop='mossy_fibers - Al'):
    """Plot differential gamma band power (ratio to mossy fibers).
    Data is already hemisphere-averaged at loading.
    """
    means = []
    stds = []
    individual_points = []
    all_data = {}
    
    for condition in desired_conditions:
        # Calculate power ratio for each trial using shared helper
        trial_ratios = compute_power_ratio(pop_name, reference_pop, condition, gamma_range)
        
        means.append(np.mean(trial_ratios))
        stds.append(np.std(trial_ratios))
        individual_points.append(trial_ratios)
        all_data[condition] = trial_ratios
    
    means = np.array(means)
    stds = np.array(stds)
    
    # Try to get FDR-corrected p-values from saved CSV
    fdr_p_values = get_fdr_p_values(title, 'Gamma Power Ratio (vs Mossy)')
    
    if fdr_p_values is not None:
        print(f"Using saved FDR-corrected p-values for {title}: {fdr_p_values}")
    else:
        print(f"No saved FDR p-values found for {title} - using raw p-values for annotations")
    
    # Create bar plot with stats (use FDR p-values if available)
    bar_plot_with_stats(
        ax, means, stds, individual_points, desired_conditions,
        bar_color=color,
        ylabel='Log₁₀(Population/Mossy Power Ratio)',
        title=title,
        add_stat_annotations=True,
        ylim_bottom=0, ylim_top=3,
        p_values=fdr_p_values  # Pass FDR-corrected p-values if available
    )
    
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    
    # Print statistical results with Cohen's d and collect results
    stats_results = print_statistical_tests(title, 'Gamma Power Ratio (vs Mossy)', 
                                            desired_conditions, individual_points)
    all_statistical_results.extend(stats_results)
    
    return all_data

# Create differential gamma band plots for both populations using viridis colors
pc_ratio_data = plot_differential_gamma_bars(pc_pop, axes[0,0], viridis_colors['teal'], 'Purkinje Cells')
cn_ratio_data = plot_differential_gamma_bars(cn_pop, axes[0,1], viridis_colors['blue'], 'Cerebellar Nuclei')

# Get coherence data (single connection per type, already hemisphere-averaged)
trig_cn_coh = get_coherence_data(connection_sets[0]['connection'], gamma_range)
sens_mot_coh = get_coherence_data(connection_sets[1]['connection'], gamma_range)

def plot_ratio_correlation(power_data, coh_data, ax, title):
    """Plot correlation between power ratio and coherence data."""
    x_vals = []
    y_vals = []
    
    for condition in desired_conditions:
        x_vals.extend(power_data[condition])
        y_vals.extend(coh_data[condition])
    
    x_vals = np.array(x_vals)
    y_vals = np.array(y_vals)
    
    r, p_val = stats.pearsonr(x_vals, y_vals)
    
    ax.scatter(x_vals, y_vals, alpha=0.6, color=viridis_colors['teal'])
    
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, p(x_line), '--', alpha=0.8, color=viridis_colors['green'])
    
    ax.set_xlabel('Log₁₀(Population/Mossy Power Ratio)')
    ax.set_ylabel('Coherence in Gamma Band')
    ax.set_title(f'{title}\nr={r:.3f}, p={p_val:.4f}')
    
    return ax

# Purkinje cell ratio correlations
plot_ratio_correlation(pc_ratio_data, trig_cn_coh, axes[1,0],
                'Cerebellar Input-Output Coherence\nvs Purkinje/Mossy Power Ratio')
plot_ratio_correlation(pc_ratio_data, sens_mot_coh, axes[2,0],
                'M1-S1 Cortex Coherence\nvs Purkinje/Mossy Power Ratio')

# CN ratio correlations
plot_ratio_correlation(cn_ratio_data, trig_cn_coh, axes[1,1],
                'Cerebellar Input-Output Coherence\nvs CN/Mossy Power Ratio')
plot_ratio_correlation(cn_ratio_data, sens_mot_coh, axes[2,1],
                'M1-S1 Cortex Coherence\nvs CN/Mossy Power Ratio')

plt.tight_layout()
save_figure_multi_format(fig, os.path.join(lesion_group_dir, 'PSD_gamma_ratio_analysis'))

# Save all statistical results to a file with FDR correction
stats_df = pd.DataFrame(all_statistical_results)

# Apply FDR correction to all p-values (both t-test and Mann-Whitney)
from statsmodels.stats.multitest import multipletests

# Get all raw p-values
raw_p_ttest = stats_df['p_value_ttest'].values
raw_p_mannwhitney = stats_df['p_value_mannwhitney'].values

# Apply Benjamini-Hochberg FDR correction to t-test p-values
_, p_ttest_fdr, _, _ = multipletests(raw_p_ttest, method='fdr_bh')
stats_df['p_value_ttest_fdr'] = p_ttest_fdr

# Apply Benjamini-Hochberg FDR correction to Mann-Whitney p-values
_, p_mannwhitney_fdr, _, _ = multipletests(raw_p_mannwhitney, method='fdr_bh')
stats_df['p_value_mannwhitney_fdr'] = p_mannwhitney_fdr

# Add significance markers based on FDR-corrected p-values
def get_significance_marker(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df['significance_fdr'] = stats_df['p_value_ttest_fdr'].apply(get_significance_marker)

# Reorder columns to have raw and FDR-corrected p-values together
cols = list(stats_df.columns)
# Move FDR columns next to their raw counterparts
new_order = ['analysis', 'measure', 'condition_1', 'condition_2', 'n_1', 'n_2', 
             'mean_1', 'mean_2', 'std_1', 'std_2', 't_statistic', 
             'p_value_ttest', 'p_value_ttest_fdr', 'significance', 'significance_fdr',
             'u_statistic', 'p_value_mannwhitney', 'p_value_mannwhitney_fdr',
             'cohens_d', 'effect_size_interpretation']
stats_df = stats_df[new_order]

stats_output_path = os.path.join(lesion_group_dir, 'statistical_comparisons.csv')
stats_df.to_csv(stats_output_path, index=False)
print(f"\n{'='*70}")
print(f"All statistical comparison results saved to: {stats_output_path}")
print(f"Total number of comparisons: {len(all_statistical_results)}")
print(f"Note: CSV includes both raw p-values and FDR-corrected p-values")
print(f"{'='*70}")

# Also save as a formatted text file for easy reading
stats_txt_path = os.path.join(lesion_group_dir, 'statistical_comparisons.txt')
with open(stats_txt_path, 'w') as f:
    f.write("STATISTICAL COMPARISONS SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    f.write("Note: FDR correction applied using Benjamini-Hochberg method\n")
    f.write("=" * 80 + "\n\n")
    
    for idx, result in stats_df.iterrows():
        f.write(f"Analysis: {result['analysis']}\n")
        f.write(f"Measure: {result['measure']}\n")
        f.write(f"Comparison: {result['condition_1']} vs {result['condition_2']}\n")
        f.write(f"  N (group 1): {result['n_1']}, N (group 2): {result['n_2']}\n")
        f.write(f"  Mean (group 1): {result['mean_1']:.4f} ± {result['std_1']:.4f}\n")
        f.write(f"  Mean (group 2): {result['mean_2']:.4f} ± {result['std_2']:.4f}\n")
        f.write(f"  T-test: t={result['t_statistic']:.3f}, p={result['p_value_ttest']:.6f} (raw) [{result['significance']}]\n")
        f.write(f"          p={result['p_value_ttest_fdr']:.6f} (FDR-corrected) [{result['significance_fdr']}]\n")
        f.write(f"  Mann-Whitney U: U={result['u_statistic']:.3f}, p={result['p_value_mannwhitney']:.6f} (raw)\n")
        f.write(f"                  p={result['p_value_mannwhitney_fdr']:.6f} (FDR-corrected)\n")
        f.write(f"  Cohen's d: {result['cohens_d']:.3f} ({result['effect_size_interpretation']} effect)\n")
        f.write("-" * 80 + "\n")
    
print(f"Formatted results also saved to: {stats_txt_path}")

plt.show()

### Power Ratio Analysis in Theta Band

In [ ]:
# Create first figure for PC and CN theta band analysis with 3x2 subplots
fig1_theta, axes1_theta = plt.subplots(3, 2, figsize=(10, 15))

# Set default font sizes
set_default_plot_params()

# Define theta band range
theta_range = (6, 12)

# Get viridis colors
viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5),
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

def plot_theta_bars(pop_name, ax, color, title):
    """Plot theta band power bars for a population.
    Data is already hemisphere-averaged at loading.
    """
    means = []
    stds = []
    individual_points = []
    all_data = {}
    
    # Get frequency values for integration
    freqs_theta = spikesPSD.sel(frequency=slice(theta_range[0], theta_range[1])).coords['frequency'].values
    
    for condition in desired_conditions:
        # Get PSD data (already hemisphere-averaged)
        condition_psd = spikesPSD.sel(
            **{'Lesion Condition': condition, 
               'Population - Region': pop_name,
               'frequency': slice(theta_range[0], theta_range[1])}
        ).values
        
        # Calculate total power using trapezoid integration for each trial
        trial_powers = np.array([np.trapz(trial_psd, freqs_theta) for trial_psd in condition_psd])
        
        means.append(np.mean(trial_powers))
        stds.append(np.std(trial_powers))
        individual_points.append(trial_powers)
        all_data[condition] = trial_powers
    
    means = np.array(means)
    stds = np.array(stds)
    
    # Try to get FDR-corrected p-values from saved CSV
    fdr_p_values = get_fdr_p_values(title, f'Theta Power ({theta_range[0]}-{theta_range[1]} Hz)')
    
    if fdr_p_values is not None:
        print(f"Using saved FDR-corrected p-values for {title}: {fdr_p_values}")
    else:
        print(f"No saved FDR p-values found for {title} - using raw p-values for annotations")
    
    # Create bar plot with stats (use FDR p-values if available)
    bar_plot_with_stats(
        ax, means, stds, individual_points, desired_conditions,
        bar_color=color, point_color=color,
        point_size=20, point_alpha=0.6,
        ylabel='Total Power in Theta Band',
        title=title,
        add_stat_annotations=True,
        p_values=fdr_p_values  # Pass FDR-corrected p-values if available
    )
    
    # Print statistical results with Cohen's d and collect results
    stats_results = print_statistical_tests(title, f'Theta Power ({theta_range[0]}-{theta_range[1]} Hz)', 
                                            desired_conditions, individual_points)
    all_statistical_results.extend(stats_results)
    
    return all_data

def get_coherence_data_theta(connection, theta_range):
    """Get coherence data in theta band organized by condition (already hemisphere-averaged)."""
    coherence_data = {}
    
    for condition in desired_conditions:
        coh_data = COH.sel(
            **{'Lesion Condition': condition,
               'Connections': connection,
               'Frequency': slice(theta_range[0], theta_range[1])}
        ).values
        
        coh_mean = np.mean(coh_data, axis=1)  # Average across frequencies
        coherence_data[condition] = coh_mean
    
    return coherence_data

def plot_theta_correlation(power_data, coh_data, ax, title):
    """Plot correlation between theta power and coherence data."""
    x_vals = []
    y_vals = []
    
    for condition in desired_conditions:
        x_vals.extend(power_data[condition])
        y_vals.extend(coh_data[condition])
    
    x_vals = np.array(x_vals)
    y_vals = np.array(y_vals)
    
    r, p_val = stats.pearsonr(x_vals, y_vals)
    
    ax.scatter(x_vals, y_vals, alpha=0.6, color=viridis_colors['yellow'])
    
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, p(x_line), '--', alpha=0.8, color=viridis_colors['teal'])
    
    ax.set_xlabel('Total Power in Theta Band')
    ax.set_ylabel('Coherence in Theta Band')
    ax.set_title(f'{title}\nr={r:.3f}, p={p_val:.4f}')
    
    return ax

# Cell population names
pc_pop = 'purkinje_cell - Al'
cn_pop = 'dcn_cell_glut_large - CN'
mossy_pop = 'mossy_fibers - Al'

# Create theta band plots for PC and CN (first row) using viridis colors
pc_theta_data = plot_theta_bars(pc_pop, axes1_theta[0,0], viridis_colors['teal'], 'Purkinje Cells Power in Theta Band')
cn_theta_data = plot_theta_bars(cn_pop, axes1_theta[0,1], viridis_colors['blue'], 'Cerebellar Nuclei Power in Theta Band')

# Get coherence data in theta band (single connection per type, already hemisphere-averaged)
trig_cn_coh_theta = get_coherence_data_theta(connection_sets[0]['connection'], theta_range)
sens_mot_coh_theta = get_coherence_data_theta(connection_sets[1]['connection'], theta_range)
print("sens_mot_coh_theta: ", sens_mot_coh_theta)

# Column 1: All Purkinje cell-related correlations
plot_theta_correlation(pc_theta_data, trig_cn_coh_theta, axes1_theta[1,0], 
                'Cerebellar Input-Output Coherence\nvs Purkinje Cell Power')
plot_theta_correlation(pc_theta_data, sens_mot_coh_theta, axes1_theta[2,0], 
                'M1-S1 Cortex Coherence\nvs Purkinje Cell Power')

# Column 2: All Cerebellar Nuclei-related correlations
plot_theta_correlation(cn_theta_data, trig_cn_coh_theta, axes1_theta[1,1], 
                'Cerebellar Input-Output Coherence\nvs Cerebellar Nuclei Power')
plot_theta_correlation(cn_theta_data, sens_mot_coh_theta, axes1_theta[2,1], 
                'M1-S1 Cortex Coherence\nvs Cerebellar Nuclei Power')

fig1_theta.suptitle('Power Ratio Analysis in Theta Band', y=1.02, fontsize=18)
plt.tight_layout()
save_figure_multi_format(fig1_theta, os.path.join(lesion_group_dir, 'PSD_theta_analysis'))

# Create second figure for Mossy Fiber analysis in theta band
fig2_theta, ax2_theta = plt.subplots(figsize=(10, 8))
mossy_theta_data = plot_theta_bars(mossy_pop, ax2_theta, viridis_colors['dark_purple'], 'Mossy Fibers Power in Theta Band')
fig2_theta.suptitle('Mossy Fibers Power in Theta Band', fontsize=18)
plt.tight_layout()
save_figure_multi_format(fig2_theta, os.path.join(lesion_group_dir, 'PSD_theta_analysis_mossy'))

plt.show()

In [ ]:
# Create figure for differential theta band analysis with 3x2 subplots
fig_theta_ratio, axes_theta_ratio = plt.subplots(3, 2, figsize=(10, 15))

# Set default font sizes
set_default_plot_params()

# Get viridis colors
viridis = cm.get_cmap('viridis')
viridis_colors = {
    'dark_purple': viridis(0.0),
    'blue': viridis(0.25),
    'teal': viridis(0.5),
    'green': viridis(0.75),
    'yellow': viridis(1.0)
}

def plot_differential_theta_bars(pop_name, ax, color, title, reference_pop='mossy_fibers - Al'):
    """Plot differential theta band power (ratio to mossy fibers).
    Data is already hemisphere-averaged at loading.
    """
    means = []
    stds = []
    individual_points = []
    all_data = {}
    
    # Get frequency values for integration
    freqs_theta = spikesPSD.sel(frequency=slice(theta_range[0], theta_range[1])).coords['frequency'].values
    
    for condition in desired_conditions:
        # Get PSD data (already hemisphere-averaged)
        pop_psd = spikesPSD.sel(
            **{'Lesion Condition': condition, 
               'Population - Region': pop_name,
               'frequency': slice(theta_range[0], theta_range[1])}
        ).values
        
        ref_psd = spikesPSD.sel(
            **{'Lesion Condition': condition, 
               'Population - Region': reference_pop,
               'frequency': slice(theta_range[0], theta_range[1])}
        ).values
        
        # Calculate integrated power for each trial
        pop_power = np.array([np.trapz(trial_psd, freqs_theta) for trial_psd in pop_psd])
        ref_power = np.array([np.trapz(trial_psd, freqs_theta) for trial_psd in ref_psd])
        
        # Calculate power ratio for each trial
        trial_ratios = np.log10(pop_power / ref_power)
        
        means.append(np.mean(trial_ratios))
        stds.append(np.std(trial_ratios))
        individual_points.append(trial_ratios)
        all_data[condition] = trial_ratios
    
    means = np.array(means)
    stds = np.array(stds)
    
    # Try to get FDR-corrected p-values from saved CSV
    fdr_p_values = get_fdr_p_values(title, 'Theta Power Ratio (vs Mossy)')
    
    if fdr_p_values is not None:
        print(f"Using saved FDR-corrected p-values for {title}: {fdr_p_values}")
    else:
        print(f"No saved FDR p-values found for {title} - using raw p-values for annotations")
    
    # Create bar plot with stats (use FDR p-values if available)
    bar_plot_with_stats(
        ax, means, stds, individual_points, desired_conditions,
        bar_color=color,
        ylabel='Log₁₀(Population/Mossy Power Ratio)',
        title=title,
        add_stat_annotations=True,
        p_values=fdr_p_values  # Pass FDR-corrected p-values if available
    )
    
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    
    # Print statistical results with Cohen's d and collect results
    stats_results = print_statistical_tests(title, 'Theta Power Ratio (vs Mossy)', 
                                            desired_conditions, individual_points)
    all_statistical_results.extend(stats_results)
    
    return all_data

# Create differential theta band plots for both populations using viridis colors
pc_theta_ratio_data = plot_differential_theta_bars(pc_pop, axes_theta_ratio[0,0], viridis_colors['teal'], 'Purkinje Cells')
cn_theta_ratio_data = plot_differential_theta_bars(cn_pop, axes_theta_ratio[0,1], viridis_colors['blue'], 'Cerebellar Nuclei')

# Get coherence data in theta band (single connection per type, already hemisphere-averaged)
trig_cn_coh_theta = get_coherence_data_theta(connection_sets[0]['connection'], theta_range)
sens_mot_coh_theta = get_coherence_data_theta(connection_sets[1]['connection'], theta_range)

def plot_theta_ratio_correlation(power_data, coh_data, ax, title):
    """Plot correlation between theta power ratio and coherence data."""
    x_vals = []
    y_vals = []
    
    for condition in desired_conditions:
        x_vals.extend(power_data[condition])
        y_vals.extend(coh_data[condition])
    
    x_vals = np.array(x_vals)
    y_vals = np.array(y_vals)
    
    r, p_val = stats.pearsonr(x_vals, y_vals)
    
    ax.scatter(x_vals, y_vals, alpha=0.6, color=viridis_colors['yellow'])
    
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, p(x_line), '--', alpha=0.8, color=viridis_colors['teal'])
    
    ax.set_xlabel('Log₁₀(Population/Mossy Power Ratio)')
    ax.set_ylabel('Coherence in Theta Band')
    ax.set_title(f'{title}\nr={r:.3f}, p={p_val:.4f}')
    
    return ax

# Purkinje cell ratio correlations
plot_theta_ratio_correlation(pc_theta_ratio_data, trig_cn_coh_theta, axes_theta_ratio[1,0],
                'Cerebellar Input-Output Coherence\nvs Purkinje/Mossy Power Ratio')
plot_theta_ratio_correlation(pc_theta_ratio_data, sens_mot_coh_theta, axes_theta_ratio[2,0],
                'M1-S1 Cortex Coherence\nvs Purkinje/Mossy Power Ratio')

# CN ratio correlations
plot_theta_ratio_correlation(cn_theta_ratio_data, trig_cn_coh_theta, axes_theta_ratio[1,1],
                'Cerebellar Input-Output Coherence\nvs CN/Mossy Power Ratio')
plot_theta_ratio_correlation(cn_theta_ratio_data, sens_mot_coh_theta, axes_theta_ratio[2,1],
                'M1-S1 Cortex Coherence\nvs CN/Mossy Power Ratio')

fig_theta_ratio.suptitle('Differential Power Ratio Analysis in Theta Band (vs Mossy Fibers)', y=1.02, fontsize=18)
plt.tight_layout()
save_figure_multi_format(fig_theta_ratio, os.path.join(lesion_group_dir, 'PSD_theta_ratio_analysis'))

# Re-save all statistical results including theta band analyses with FDR correction
stats_df = pd.DataFrame(all_statistical_results)

# Apply FDR correction to all p-values (both t-test and Mann-Whitney)
from statsmodels.stats.multitest import multipletests

# Get all raw p-values
raw_p_ttest = stats_df['p_value_ttest'].values
raw_p_mannwhitney = stats_df['p_value_mannwhitney'].values

# Apply Benjamini-Hochberg FDR correction to t-test p-values
_, p_ttest_fdr, _, _ = multipletests(raw_p_ttest, method='fdr_bh')
stats_df['p_value_ttest_fdr'] = p_ttest_fdr

# Apply Benjamini-Hochberg FDR correction to Mann-Whitney p-values
_, p_mannwhitney_fdr, _, _ = multipletests(raw_p_mannwhitney, method='fdr_bh')
stats_df['p_value_mannwhitney_fdr'] = p_mannwhitney_fdr

# Add significance markers based on FDR-corrected p-values
def get_significance_marker(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df['significance_fdr'] = stats_df['p_value_ttest_fdr'].apply(get_significance_marker)

# Reorder columns
new_order = ['analysis', 'measure', 'condition_1', 'condition_2', 'n_1', 'n_2', 
             'mean_1', 'mean_2', 'std_1', 'std_2', 't_statistic', 
             'p_value_ttest', 'p_value_ttest_fdr', 'significance', 'significance_fdr',
             'u_statistic', 'p_value_mannwhitney', 'p_value_mannwhitney_fdr',
             'cohens_d', 'effect_size_interpretation']
stats_df = stats_df[new_order]

stats_output_path = os.path.join(lesion_group_dir, 'statistical_comparisons.csv')
stats_df.to_csv(stats_output_path, index=False)
print(f"\n{'='*70}")
print(f"All statistical results (including theta) saved to: {stats_output_path}")
print(f"Total number of comparisons: {len(all_statistical_results)}")
print(f"Note: CSV includes both raw p-values and FDR-corrected p-values")
print(f"{'='*70}")

# Also save as a formatted text file for easy reading
stats_txt_path = os.path.join(lesion_group_dir, 'statistical_comparisons.txt')
with open(stats_txt_path, 'w') as f:
    f.write("STATISTICAL COMPARISONS SUMMARY (including Theta Band)\n")
    f.write("=" * 80 + "\n\n")
    f.write("Note: FDR correction applied using Benjamini-Hochberg method\n")
    f.write("=" * 80 + "\n\n")
    
    for idx, result in stats_df.iterrows():
        f.write(f"Analysis: {result['analysis']}\n")
        f.write(f"Measure: {result['measure']}\n")
        f.write(f"Comparison: {result['condition_1']} vs {result['condition_2']}\n")
        f.write(f"  N (group 1): {result['n_1']}, N (group 2): {result['n_2']}\n")
        f.write(f"  Mean (group 1): {result['mean_1']:.4f} ± {result['std_1']:.4f}\n")
        f.write(f"  Mean (group 2): {result['mean_2']:.4f} ± {result['std_2']:.4f}\n")
        f.write(f"  T-test: t={result['t_statistic']:.3f}, p={result['p_value_ttest']:.6f} (raw) [{result['significance']}]\n")
        f.write(f"          p={result['p_value_ttest_fdr']:.6f} (FDR-corrected) [{result['significance_fdr']}]\n")
        f.write(f"  Mann-Whitney U: U={result['u_statistic']:.3f}, p={result['p_value_mannwhitney']:.6f} (raw)\n")
        f.write(f"                  p={result['p_value_mannwhitney_fdr']:.6f} (FDR-corrected)\n")
        f.write(f"  Cohen's d: {result['cohens_d']:.3f} ({result['effect_size_interpretation']} effect)\n")
        f.write("-" * 80 + "\n")
    
print(f"Formatted results also saved to: {stats_txt_path}")

plt.show()